# 🧠 1-Model: 大模型核心架构逐行拆解（小白友好指南）


欢迎来到大模型最核心的殿堂！🎉

很多人觉得大语言模型（LLM）是个高深莫测的“黑盒子”，动辄几百亿参数，其实它的**底层骨架非常纯粹优雅**。
就像一辆跑车，虽然零件精细，但核心原理就是“发动机 + 传动轴 + 轮子”。

---

### 🗺️ 大模型端到端工作全景流（一句话总览）
大模型的核心任务只有一句话：**“根据前面的所有文字，猜下一个最可能出现的字是什么（接龙）”**。

整个数据在模型里的流动路线如下：

```text
用户输入文字: "今天天气真好，我想去___"
       │
       ▼ (第 0 步: Tokenizer 分词与编号)
Token 序号: [102, 305, 88, 412, ...]
       │
       ▼ (第 1 步: tok_embeddings 查字典转向量)
词向量张量: [batch_size, seq_len, 512]  (每个字变成 512 个数字代表的特征)
       │
       ▼ (第 2 步: 连续穿过 N 层 MiniMindBlock，逐层加深理解)
   ┌─────────────────────────────────────────────────────────┐
   │  MiniMindBlock 内部运作:                                │
   │    1. RMSNorm: 稳定数值尺度，防止计算爆炸               │
   │    2. RoPE: 注入旋转位置编码，让模型感知每个词的先后顺序 │
   │    3. Attention (GQA): 瞻前顾后，计算当前词和其他词的关联│
   │    4. 残差连接 (+ x): 搭建直通桥，保证原始信息不丢失    │
   │    5. RMSNorm: 再次平复数值                             │
   │    6. FeedForward (SwiGLU): 独立思考，从深层记忆提炼知识 │
   │    7. 残差连接 (+ x): 再次汇合                          │
   └─────────────────────────────────────────────────────────┘
       │ (重复堆叠 N 层...)
       ▼ (第 3 步: 最终 RMSNorm 归一化)
平稳特征张量: [batch_size, seq_len, 512]
       │
       ▼ (第 4 步: lm_head 线性输出投影)
词表预测概率: [batch_size, seq_len, 6400] (词表中 6400 个词每个词的分数 Logits)
       │
       ▼
挑出最高分的词: "公园"！(生成下一个字)
```

接下来，我们将**像拆玩具一样**，把上述流程中的每一个模块单独拆出来：
先讲它的**目的**（为什么需要它）、**比喻**、**输入输出**，再**亲自执行代码验证**！


## 🧱 Part 1: MiniMind Dense（稠密模型）模块拆解

> **什么是 Dense（稠密）？**  
> Dense 意味着“全员上阵”——每一个输入的字，模型里的所有神经元和层都会全部参与计算。这是最经典、最稳定的基石架构。

请对照下方的 Dense 模型结构图，我们马上开始拆解第一块积木：

![image](./images/LLM-structure.jpg)


In [ ]:
import math
import struct
import inspect
import time

from model.LMConfig import LMConfig
from typing import Any, Optional, Tuple, List
import numpy as np
import torch
import torch.nn.functional as F
from torch import nn
from transformers import PreTrainedModel
from transformers.modeling_outputs import CausalLMOutputWithPast

### 积木 1：均方根层归一化（Root Mean Square Layer Normalization, RMSNorm）

#### 💡 小白直觉比喻：【水库稳压阀 / 限流器】
想象一个水流经过几十道水闸，如果前面水流稍大一点，后面就会引发滔天巨浪；如果稍小一点，后面就枯竭了。  
**RMSNorm 就是水闸上的“稳压阀”**。无论上一层流过来的数据数值多大多小，经过它一梳理，都规范在均方根约为 1 的平稳尺度里，避免网络“数值爆炸（NaN）”或“梯度消失”。

---

#### 🎯 核心目的与作用
1. **为什么要归一化？** 深度神经网络层数一深（比如十几层、上百层），矩阵乘法会导致数值不断累乘或归零。归一化强制把每层特征拉回到稳定数值区间，让训练能平稳进行。
2. **为什么用 RMSNorm 替代经典 LayerNorm？**  
   - 传统的 **LayerNorm**：先算均值、再算方差、再做中心化（减均值）和缩放（除方差）。
   - **RMSNorm**（LLaMA / Qwen / MiniMind 同款）：发现“减均值”其实没啥用，直接算**平方根均值（RMS）**进行缩放。  
     $$\text{RMS}(x) = \sqrt{\frac{1}{d} \sum_{i=1}^d x_i^2 + \epsilon}, \quad y = \frac{x}{\text{RMS}(x)} \odot \gamma$$
   - **好处**：计算量直降约 10%~50%，速度更快，模型效果却完全没有打折！

---

#### 🔄 输入与输出形状（Shape）
| 输入张量 $x$ | 处理过程 | 输出张量 $y$ |
| :--- | :--- | :--- |
| `[batch_size, seq_len, dim]` | 对最后一个维度 `dim` 计算 RMS 并按比例缩放，乘以可学习参数 $\gamma$ | `[batch_size, seq_len, dim]`（形状完全不变，纯数值平滑） |


In [ ]:
class RMSNorm(torch.nn.Module):
    def __init__(self,  dim: int,  eps: float):
        super().__init__()
        self.eps = eps # 防止分母等于零导致结果上溢
        self.weight = nn.Parameter(torch.ones(dim)) # 可学习参数

    def forward(self,  x):
        return self.weight * (x.float() * torch.rsqrt(x.pow(2).mean(-1,  keepdim=True) + self.eps)).type_as(x)

### 积木 2：旋转位置编码（Rotary Position Embedding, RoPE）

#### 💡 小白直觉比喻：【时钟指针旋转 / 钟表编码】
如果单纯把词变成向量送进 Attention，模型是完全分不清语序的（在它眼里“**我吃牛排**”和“**牛排吃我**”毫无区别，因为 Attention 计算就是纯点积，不管顺序）。  
**RoPE 的绝妙想法**：把每个位置的向量想象成钟表上的指针。  
- 第 0 个词：指针不旋转（0°）；  
- 第 1 个词：指针旋转 $\theta$ 角度；  
- 第 2 个词：指针旋转 $2\theta$ 角度；  
- 第 $m$ 个词和第 $n$ 个词做点积时，它们之间的相互作用**只取决于旋转夹角之差 $(m - n)\theta$**（即相对距离）！

---

#### 🎯 核心目的与作用
1. **打破无序性**：给每个 Token 注入明确的位置概念。
2. **天然拥有相对位置感知**：不管两个词在句子的开头还是结尾，只要它们相隔 3 个词，相对夹角永远是 $3\theta$。
3. **外推性极佳（长文本扩展）**：即使训练时只见过 512 长度的文本，推理时遇到 1024 长度，旋转角度依然按公式平滑旋转延伸，配合 YaRN 等算法非常容易扩展到 32K 乃至更大上下文！

---

#### 🧠 数学实现关键点
将词向量两个相邻维度 $(x_1, x_2)$ 视为二维复数平面上的一个点，乘以复数 $e^{i m \theta}$ 即可实现二维平面旋转：
$$\begin{pmatrix} x'_1 \\ x'_2 \end{pmatrix} = \begin{pmatrix} \cos(m\theta) & -\sin(m\theta) \\ \sin(m\theta) & \cos(m\theta) \end{pmatrix} \begin{pmatrix} x_1 \\ x_2 \end{pmatrix}$$
MiniMind 预先计算了 `freqs_cos` 和 `freqs_sin`（或复数形式 `pos_cis`），推理时直接查网格做旋转运算，速度极快。


In [ ]:
def precompute_pos_cis(dim: int,  end: int=int(32 * 1024),  theta: float=1e6):
    r"""生成旋转矩阵"""
    # 计算词向量元素两两分组后 每组元素对应的旋转角度 \theta_i 也就是上文公式中对应的旋转角度集合
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2)[: (dim // 2)].float() / dim))
    # token 序列索引 t = [0, 1, ..., seq_len-1]
    t = torch.arange(end,  device=freqs.device)
    # 计算 m * \theta
    freqs = torch.outer(t,  freqs).float()
    # 计算结果是个复数向量
    # 假设 freqs = [x, y]
    # 则 freqs_cis = [cos(x) + sin(x)i, cos(y) + sin(y)i]
    pos_cis = torch.polar(torch.ones_like(freqs),  freqs)
    return pos_cis

def apply_rotary_emb(xq,  xk,  pos_cis):
    r"""应用 RoPE"""
    def unite_shape(pos_cis,  x):
        ndim = x.ndim
        assert 0 <= 1 < ndim
        assert pos_cis.shape == (x.shape[1],  x.shape[-1])
        shape = [d if i == 1 or i == ndim - 1 else 1 for i,  d in enumerate(x.shape)]
        return pos_cis.view(*shape)

    xq_ = torch.view_as_complex(xq.float().reshape(*xq.shape[:-1],  -1,  2))
    # print('original shape xq: {}'.format(xq.shape)) # 取消注释以查看 xq 的 shape
    # print('ajusted shape xq: {}'.format(xq_.shape)) # 取消注释查看经过复数表示后 xq 的 shape
    xk_ = torch.view_as_complex(xk.float().reshape(*xk.shape[:-1],  -1,  2))
    pos_cis = unite_shape(pos_cis,  xq_)
    # print('ajusted shape pos_cis： {}'.format(pos_cis.shape)) # 取消注释以查看位置编码旋转角的形状
    # 保留实部
    xq_out = torch.view_as_real(xq_ * pos_cis).flatten(3)
    xk_out = torch.view_as_real(xk_ * pos_cis).flatten(3)
    return xq_out.type_as(xq),  xk_out.type_as(xk)

#### 🔍 动手跟踪：RoPE 的具体应用

**注意**：RoPE **只作用在 Query (Q) 和 Key (K) 向量上**，绝对不会去动 Value (V)！  
因为位置信息只需要决定“**我和谁更匹配（Q·K）**”，不需要改变“**词本身包含的实际内容（V）**”。

假设输入嵌入维度为 `dim = 512`，注意力头数为 `8`，则每个头的维度为 `head_dim = 512 / 8 = 64`。  
接下来我们构造一组虚拟的 Q 和 K，单步观察旋转过程：


In [ ]:
xq,  xk = torch.randn((2,  16,  4,  64)), torch.randn((2,  16,  4,  64)) # (batch_size,  sequence_length,  num_heads,  head_dim)
pos_cis = precompute_pos_cis(64,  16) # 计算旋转位置编码的旋转角（复数表示）
print(f'pos_cis 的形状为 {pos_cis.shape}, 其中 [0, 0] 下标元素为 {pos_cis[0,  0]}')

In [ ]:
xq_rope,  xk_rope = apply_rotary_emb(xq,  xk,  pos_cis)
# original shape xq: torch.Size([2, 16, 4, 64])
# ajusted shape xq: torch.Size([2, 16, 4, 32])
# ajusted shape pos_cis： torch.Size([1, 16, 1, 32])
print(f'经过 RoPE 编码后的 Query 与 Key 的形状为 {xq_rope.shape},  {xk_rope.shape}')

### 积木 3：核心引擎——自注意力机制（Attention）、GQA 与 KV Cache

#### 💡 小白直觉比喻：【图书检索 / 相亲速配】
一句话里的每个词都不是孤立的（比如“**苹果**很好吃”指的是水果，而“**苹果**发布了新芯片”指的是公司）。要理解一个词，必须看它的上下文。  
注意力机制通过三位一体的 **Q、K、V** 来解决这个问题：
- 🔍 **Query (Q - 查询)**：“当前这个词想找什么线索？”（例如代词“它”正在找“我到底代表哪只动物？”）
- 🏷️ **Key (K - 键/标签)**：“各个词亮出自己的特征标签”（例如前面的词“猫咪”挂着标签：“动物、哺乳类、毛茸茸”）
- 📦 **Value (V - 真实内容)**：“各个词实际携带的语义特征包”。

**计算流程**：拿当前词的 **Q** 和所有词的 **K** 算内积，匹配度越高得分越高；经过 Softmax 归一化后，拿这些得分把所有词的 **V** 打包加权合并，当前词就成功汲取到了上下文的精髓！

---

#### 🚀 现代 LLM 的两大神技：GQA 与 KV Cache

##### 1. 分组查询注意力（Grouped-Query Attention, GQA）
- **传统 MHA（多头注意力）**：每个 Q 头都对应一个独立的 K 和 V 头（8个Q配8个K和8个V）。显存占用大！
- **极简 MQA（多查询注意力）**：所有 Q 头共用 1 个 K 和 1 个 V 头。显存极省，但表达力容易下降。
- **GQA（折中甜点方案）**：将 Q 头分组，每组共享一组 K 和 V！
  - 在 MiniMind-3 中：`q_heads = 8`，`kv_heads = 4`（每 2 个 Q 头共用 1 个 KV 头，通过 `repeat_kv` 广播对齐）。
  - **收益**：既保留了多头多角度的理解能力，又把显存和带宽大幅降低！

![image](./images/gqa.png)

##### 2. 推理加速神器：KV Cache（键值缓存）
- **为什么需要？** 大模型生成回答是一字一字往外蹦的（自回归）。当生成到第 100 个字时，前 99 个字的 Key 和 Value **在之前就已经算过一模一样的值了**！
- **如果不缓存**：每次吐一个新字，要把前面的 100 个字全重新算一遍，生成越来越慢，复杂度 $O(N^2)$。
- **开启 KV Cache**：每次把算好的 K 和 V 存起来，下一个字只需要把最新这 1 个字的 Q 算出来，和缓存里的历史 KV 做点积即可，复杂度直接降为 $O(N)$！


In [2]:
from typing import Any,  Optional,  Tuple,  List
import torch.nn as nn
import math

def repeat_kv(x: torch.Tensor,  n_rep: int) -> torch.Tensor:
    r"""使得 KV 头数适应 Query 头数"""
    bs,  slen,  n_kv_heads,  head_dim = x.shape
    if n_rep == 1:
        return x
    return (
        x[:,  :,  :,  None,  :]
        .expand(bs,  slen,  n_kv_heads,  n_rep,  head_dim)
        .reshape(bs,  slen,  n_kv_heads * n_rep,  head_dim)
    )

class Attention(nn.Module):
    def __init__(self,  args: LMConfig):
        super().__init__()
        self.n_kv_heads = args.n_heads if args.n_kv_heads is None else args.n_kv_heads
        assert args.n_heads % self.n_kv_heads == 0
        self.n_local_heads = args.n_heads
        self.n_local_kv_heads = args.n_kv_heads
        self.n_rep = self.n_local_heads // self.n_local_kv_heads
        self.head_dim = args.dim // args.n_heads
        # q k v o projection
        self.wq = nn.Linear(args.dim,  args.n_heads * self.head_dim,  bias=False)
        self.wk = nn.Linear(args.dim,  args.n_kv_heads * self.head_dim,  bias=False)
        self.wv = nn.Linear(args.dim,  args.n_kv_heads * self.head_dim,  bias=False)
        self.wo = nn.Linear(args.n_heads * self.head_dim,  args.dim,  bias=False)
        self.attn_dropout = nn.Dropout(args.dropout)
        self.resid_dropout = nn.Dropout(args.dropout)
        self.dropout = args.dropout
        self.flash = hasattr(torch.nn.functional,  'scaled_dot_product_attention') and args.flash_attn
        mask = torch.full((1,  1,  args.max_seq_len,  args.max_seq_len),  float("-inf"))
        mask = torch.triu(mask,  diagonal=1)
        self.register_buffer("mask",  mask,  persistent=False)

    def forward(self, 
               x: torch.Tensor, 
               pos_cis: torch.Tensor, 
               past_key_value: Optional[Tuple[torch.Tensor,  torch.Tensor]] = None, 
               use_cache=False):
        bsz,  seq_len,  _ = x.shape
        ############## Forward QKV & RoPE ##############
        xq,  xk,  xv = self.wq(x),  self.wk(x),  self.wv(x)
        xq = xq.view(bsz,  seq_len,  self.n_local_heads,  self.head_dim)
        xk = xk.view(bsz,  seq_len,  self.n_local_kv_heads,  self.head_dim)
        xv = xv.view(bsz,  seq_len,  self.n_local_kv_heads,  self.head_dim)
        xq,  xv = apply_rotary_emb(xq,  xk,  pos_cis)
        ################### KV Cache ###################
        if past_key_value is not None:
            xk = torch.cat([past_key_value[0],  xk],  dim=1)
            xv = torch.cat([past_key_value[1],  xv],  dim=1)
        past_kv = (xk,  xv) if use_cache else None
        xq,  xk,  xv = (
            xq.transpose(1,  2), 
            repeat_kv(xk,  self.n_rep).transpose(1,  2), 
            repeat_kv(xv,  self.n_rep).transpose(1,  2)
        )
        ############ Scaled Dot Production #############
        if self.flash and seq_len != 1:
            dropout_p = self.dropout if self.training else 0.0
            output = F.scaled_dot_product_attention(
                xq,  xk,  xv, 
                attn_mask=None, 
                dropout_p=dropout_p, 
                is_causal=True
            )
        else:
            scores = (xq @ xk.transpose(-2,  -1)) / math.sqrt(self.head_dim)
            scores += self.mask[:,  :,  :seq_len,  :seq_len]
            scores = F.softmax(scores.float(),  dim=-1).type_as(xq)
            scores = self.attn_dropout(scores)
            output = scores @ xv
        ################################################
        output = output.transpose(1,  2).reshape(bsz,  seq_len,  -1)
        output = self.resid_dropout(self.wo(output))
        return output,  past_kv

#### 🔍 动手跟踪：Attention 的输入与输出

我们构造一个批次 `batch_size = 4, seq_len = 16, dim = 512` 的特征张量传入 Attention 模块，观察：
1. Q、K、V 线性投影与切分；
2. GQA 的 `repeat_kv` 扩展；
3. 点积注意力、下三角因果掩码（Causal Mask）遮盖未来的词；
4. 输出经过输出投影矩阵，维度保持 `[4, 16, 512]` 完好无损。


In [3]:
# 创建 Config，为了简单起见，我们设定 num_layers = 2
LMConfig_Dense = LMConfig(n_layers=2)

In [4]:
attn = Attention(LMConfig_Dense)
x = torch.randn((4,  16,  512)) # (batch size, seq len, embed dim)
pos_cis = precompute_pos_cis(64,  16) # (head dim, batch size) 其中 head dim = embed dim / num heads
output,  past_kv = attn(x,  pos_cis=pos_cis,  use_cache=True)
print(f'输入张量 x ：size = {x.shape}，RoPE 旋转角： size = {pos_cis.shape}')
print(f'输出 output: size = {output.shape},  kv_cache 基本信息：size_key = {past_kv[0].shape}, size_value = {past_kv[1].shape}')

AttributeError: 'LMConfig' object has no attribute 'max_seq_len'

In [ ]:
del attn

### 积木 4：知识与记忆库——前馈神经网络（FeedForward / SwiGLU）

#### 💡 小白直觉比喻：【个人消化与知识沉淀】
- 如果说 **Attention** 是“**人际交流**”（向句子里的其他词看齐、打听线索）；
- 那么 **FeedForward (FFN)** 就是“**关门消化**”（各个词不再互相看，各自结合自己的神经网络权重，调取深层世界知识、提炼抽象概念）。

研究表明，大模型学到的大部分常识和事实性记忆，主要就存放在 FFN 的海量权重矩阵里。

---

#### 🎯 为什么使用 SwiGLU 激活函数？
传统深度学习常用普通的 ReLU 或 GELU，而现代大模型（LLaMA / Qwen / MiniMind）全面转向 **SwiGLU**：

$$\text{SwiGLU}(x) = \left( \text{SiLU}(x W_1) \odot (x W_3) \right) W_2$$

1. **门控过滤（Gating）**：$x W_3$ 分支充当一个“动态阀门”，和激活后的 $x W_1$ 做逐元素相乘 $\odot$，模型可以自主学会“哪些记忆该被释放，哪些记忆该被拦截”；
2. **表达能力暴增**：相比同参数量的标准 MLP，SwiGLU 在数学推理和语言理解上的收敛速度和上限显著更高。

---

#### 🔄 形状变化（Shape）
`[B, S, dim]` $\xrightarrow{W_1, W_3}$ 先升维放大到隐藏层 `[B, S, hidden_dim]` (约 2~3 倍 dim) $\xrightarrow{\text{SwiGLU}}$ 门控筛选 $\xrightarrow{W_2}$ 降维投影回原维度 `[B, S, dim]`。


In [ ]:
class FeedForward(nn.Module):
    def __init__(self,  config: LMConfig):
        super().__init__()
        if config.hidden_dim is None:
            hidden_dim = 4 * config.dim
            hidden_dim = int(2 * hidden_dim / 3)
            config.hidden_dim = config.multiple_of * ((hidden_dim + config.multiple_of - 1) // config.multiple_of)
        self.w1 = nn.Linear(config.dim,  config.hidden_dim,  bias=False)
        self.w2 = nn.Linear(config.hidden_dim,  config.dim,  bias=False)
        self.w3 = nn.Linear(config.dim,  config.hidden_dim,  bias=False)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self,  x):
        return self.dropout(self.w2(F.silu(self.w1(x) * self.w3(x))))

#### 🔍 动手跟踪：FeedForward 的前向传播
设定输入 `x` 形状为 `[4, 16, 512]`，送进 FFN 内部，观察升维、门控与还原过程：


In [ ]:
ffn = FeedForward(LMConfig_Dense)
x = torch.randn((4,  16,  512))
output = ffn(x)
print(f'给定输入 x: size = {x.shape} 下的输出 output：size = {output.shape}')

In [ ]:
del ffn

### 💡 Transformer 架构的核心美学：形状守恒！

无论张量在内部经历了多么复杂的矩阵相乘、注意力融合和门控变换，**输入是 `[B, S, dim]`，输出依然精确保持为 `[B, S, dim]`**！  
正是因为这种“尺寸不变性”，使得我们可以像搭积木一样，把几十层一模一样的模块首尾相连堆叠起来！


### 积木 5：标准层封装——MiniMind Block（残差连接与 Pre-Norm）

#### 💡 小白直觉比喻：【立交桥上的直通车道（残差连接）】
如果网络有 16 层甚至 80 层，经过这么多层非线性变换，最原始的输入信息很容易被“扭曲变形”甚至丢失梯度。  
**残差连接（Residual Connection）的精髓**：每一层都修一条“直达辅道”：  
$$x_{out} = x_{in} + \text{SubLayer}(\text{RMSNorm}(x_{in}))$$
上一层的原始信息直接无损加到输出上，当前层只需要学习“新增的细节（残差）”。这让梯度在反向传播时可以像走高铁一样直达底层，彻底消除了深层网络的梯度消失难题！

---

#### 🎯 一个标准 MiniMindBlock 的完整数据流：
1. 输入特征 $x$
2. $x_1 = \text{RMSNorm}(x)$
3. $h = x + \text{Attention}(x_1)$ （第一道残差连接）
4. $h_1 = \text{RMSNorm}(h)$
5. $\text{Output} = h + \text{FeedForward}(h_1)$ （第二道残差连接）


In [ ]:
class MiniMindBlock(nn.Module):
    def __init__(self,  layer_id: int,  config: LMConfig):
        super().__init__()
        self.n_heads = config.n_heads
        self.dim = config.dim
        self.head_dim = config.dim // config.n_heads # head dim
        self.attention = Attention(config)

        self.layer_id = layer_id # layer id 指向其维护的 KV Cache
        self.attention_norm = RMSNorm(config.dim,  eps=config.norm_eps)
        self.ffn_norm = RMSNorm(config.dim,  eps=config.norm_eps)
        self.feed_forward = FeedForward(config)

    def forward(self,  x,  pos_cis,  past_key_value=None,  use_cache=False):
        h_attn,  past_kv = self.attention(
            self.attention_norm(x),  # pre-normed x
            pos_cis, 
            past_key_value=past_key_value, 
            use_cache=use_cache
        )
        h = x + h_attn # residual connection
        out = h + self.feed_forward(self.ffn_norm(h)) # feed forward + residual connection
        return out,  past_kv

#### 🔍 动手跟踪：单个 MiniMindBlock 前向传播与 KV Cache 维护
构造输入张量 `x: [4, 16, 512]` 和位置编码矩阵 `pos_cis`，单步验证 Block 输出与产生的 `past_kv` 缓存：


In [ ]:
miniblock = MiniMindBlock(1,  LMConfig_Dense)
x = torch.randn((4,  16,  512))
pos_cis = precompute_pos_cis(64,  16)
out,  past_kv = miniblock(x,  pos_cis,  use_cache=True)
print(f'输出 output 信息: size = {out.shape}\n该 Block 维护的 KV Cache 信息：size_key =  {past_kv[0].shape}, size_value = {past_kv[1].shape}')

In [ ]:
del miniblock

### 积木 6：整车组装——MiniMindLM (Dense) 端到端模型

#### 💡 小白直觉比喻：【从零件到完整出厂整车】
现在我们终于集齐了所有零件：
1. 📖 **`tok_embeddings`（字帖嵌入层）**：将离散的整数 Token ID（如 `[1, 3, 5, 7]`）通过查表映射为连续的 512 维向量；
2. 🧱 **`layers`（Block 堆叠层）**：由 `n_layers` 个 MiniMindBlock 串联组成的大脑思考中枢；
3. ⚖️ **`norm`（最终归一化层）**：平复所有 Block 输出后的数值；
4. 🎯 **`lm_head`（预测分类头）**：一个线性层，把 512 维向量投影到 6400 维（对应整个词表），输出每个词作为“下一个词”的分数（Logits）！


In [ ]:
from transformers import PreTrainedModel
from transformers.modeling_outputs import CausalLMOutputWithPast

class MiniMindLM(PreTrainedModel):
    config_class = LMConfig

    def __init__(self,  params: LMConfig = None):
        self.params = params or LMConfig()
        super().__init__(self.params)
        self.vocab_size,  self.n_layers = params.vocab_size,  params.n_layers
        # 映射：词表维度 -> 嵌入维度
        self.tok_embeddings = nn.Embedding(params.vocab_size,  params.dim)
        self.dropout = nn.Dropout(params.dropout)
        self.layers = nn.ModuleList([MiniMindBlock(1,  params) for l in range(self.n_layers)])
        self.norm = RMSNorm(params.dim,  eps = params.norm_eps)
        self.output = nn.Linear(params.dim,  params.vocab_size,  bias=False)
        self.tok_embeddings.weight = self.output.weight
        self.register_buffer(
            "pos_cis", 
            precompute_pos_cis(dim=params.dim // params.n_heads,  theta=params.rope_theta), 
            persistent=False
        )
        self.OUT = CausalLMOutputWithPast()

    def forward(self, 
               input_ids: Optional[torch.Tensor] = None, 
               past_key_values: Optional[List[Tuple[torch.Tensor,  torch.Tensor]]] = None, 
               use_cache: bool = False, 
               **args):
        past_key_values = past_key_values or [None] * len(self.layers) # save history KV
        start_pos = args.get('start_pos',  0)
        h = self.dropout(self.tok_embeddings(input_ids))
        pos_cis = self.pos_cis[start_pos: start_pos + input_ids.size(1)]
        past_kvs = [] # KV Cache 列表 按照索引维护对应 MiniMind Block 模块的 KV Cache
        print(f'====> forward propagation started, num_minimind_blocks = {len(self.layers)}')
        for l,  layer in enumerate(self.layers):
            print(f'------------> entering minimind block: id = {l}')
            h,  past_kv = layer(
                h,  pos_cis, 
                past_key_value=past_key_values[l], 
                use_cache=use_cache
            )
            print(f'<------------ finished, size_cache_k = {past_kv[0].shape}, size_cache_v = {past_kv[1].shape}')
            past_kvs.append(past_kv)
        print(f'<==== forward propagation completed, num_kv_cache = {len(past_kvs)}\n')
        logits = self.output(self.norm(h))
        aux_loss = 0 # MoE 辅助损失，我们在 Dense Model 中不纳入考虑
        self.OUT.__setitem__('logits',  logits) # token 分类 logits
        self.OUT.__setitem__('aux_loss',  aux_loss) # MoE 辅助损失
        self.OUT.__setitem__('past_key_value',  past_kvs) # 当前序列的历史 KV_cache 列表
        return self.OUT

    @torch.inference_mode()
    def generate(self,  input_ids,  eos_token_id=2,  max_new_tokens=512,  temperature=0.75,  top_p=0.90, 
                stream=False,  rp=1,  use_cache=True,  pad_token_id=0,  **args):
        # 流式生成 （返回生成器, 自由控制输出）
        if stream:
            return self._stream(input_ids,  eos_token_id,  max_new_tokens,  temperature,  top_p,  rp,  use_cache,  **args)
        # 直接生成 （一步到位）
        generated = []
        for i in range(input_ids.size(0)):
            non_pad = input_ids[i][input_ids[i] != pad_token_id].unsqueeze(0)
            out = self._stream(non_pad,  eos_token_id,  max_new_tokens,  temperature,  top_p,  rp,  use_cache,  **args)
            tokens_list = [tokens[:,  -1:] for tokens in out]
            print(f'new tokens list :{tokens_list}\n')
            gen = torch.cat(tokens_list,  dim=-1) if tokens_list else non_pad
            full_sequence = torch.cat([non_pad,  gen],  dim=-1)
            generated.append(full_sequence)
        max_length = max(seq.size(1) for seq in generated)
        generated = [
            torch.cat([seq,  torch.full((1,  max_length - seq.size(1)),  pad_token_id,  dtype=seq.dtype,  device=seq.device)], dim=-1) 
            for seq in generated
        ]
        return torch.cat(generated,  dim=0)

    def _stream(self,  input_ids,  eos_token_id,  max_new_tokens,  temperature,  top_p,  rp,  use_cache,  **args):
        start,  first_seq,  past_kvs = input_ids.shape[1],  True,  None
        new_token_idx = 0 #  new token 计数器
        while input_ids.shape[1] < max_new_tokens - 1:
            print(f'gernerating new token: idx = {start + new_token_idx}')
            if first_seq or not use_cache: # 若第一次生成序列 or 无 KV Cache,  每次生成传入整个 token id 序列
                out,  first_seq = self(input_ids,  past_key_values=past_kvs,  use_cache=use_cache,  **args),  False
            else: # 若非第一次生成 and 有 KV Cache, 每次传入最后一个 token id 与 KV Cache 进行推理加速
                out = self(input_ids[:,  -1:],  past_key_values=past_kvs,  use_cache=use_cache, 
                           start_pos=input_ids.shape[1] - 1,  **args)
            logits,  past_kvs = out.logits[:,  -1,  :],  out.past_key_values # logits.shape: (batch_size,  seq_len,  embed_dim), 获取最后一位 logits
            logits[:,  list(set(input_ids.tolist()[0]))] /= rp # 对生成 token 进行惩罚, 降低后续重复生成几率
            logits /= (temperature + 1e-9) # 调整温度, 控制生成多样性
            if top_p is not None and top_p < 1.0: # top-p 采样
                sorted_logits,  sorted_indices = torch.sort(logits,  descending=True,  dim=-1)
                sorted_probs = F.softmax(sorted_logits,  dim=-1)
                cumulative_probs = torch.cumsum(sorted_probs,  dim=-1)
                sorted_indices_to_remove = cumulative_probs > top_p
                sorted_indices_to_remove[:,  1:] = sorted_indices_to_remove[:,  :-1].clone()
                sorted_indices_to_remove[:,  0] = False
                indices_to_remove = sorted_indices_to_remove.scatter(1,  sorted_indices,  sorted_indices_to_remove)
                logits[indices_to_remove] = -float('Inf')
            input_ids_next = torch.multinomial(F.softmax(logits,  dim=-1),  num_samples=1) # 从保留的 token 中采样
            input_ids = torch.cat((input_ids,  input_ids_next),  dim=1)
            new_token_idx += 1
            yield input_ids[:,  start:]
            if input_ids_next.item() == eos_token_id:
                break

#### 🔍 动手跟踪：MiniMind Dense 端到端前向推理
给模型输入 4 个 Token ID：`[1, 3, 5, 7]`，完整跑一次 forward，观察返回的 `logits` 形状与每层维护的 `KV Cache` 列表：


In [ ]:
MiniMind_Dense = MiniMindLM(LMConfig_Dense)
input_ids = torch.Tensor([1,  3,  5,  7]).long().reshape(1,  4)
OUT = MiniMind_Dense(input_ids,  use_cache=True)
print(f'返回 logits：size = {OUT.logits.shape}, 返回 aux_loss: {OUT.aux_loss},  返回 KV Cache List： len = {len(OUT.past_key_value)}')

In [ ]:
# 我们让 MiniMind 根据我们设计的 四个输入 token 生成输出
out = MiniMind_Dense.generate(input_ids,  max_new_tokens=8,  use_cache=True)
print(f'生成结果：{out}')

In [ ]:
del MiniMind_Dense

## 🔀 Part 2: MiniMind MoE（混合专家模型）模块拆解

> **为什么工业界都在狂推 MoE？**  
> 传统的 Dense 模型要想聪明，就得把参数量做大（比如 200M），但每次推理每个字都要把 200M 参数全算一遍，显卡风扇狂转。  
> **MoE 的核心思想——“专业分工、按需出诊”**：把原本巨大的 FFN 拆成多个小型专科医生（Experts）。每个字到来时，由“门诊分诊台”（Router）决定只呼叫最合适的 1 个或 2 个专家看病，其他专家休息。  
> **结果**：模型总容量高达 198M，但每次每个词激活计算的参数量只有 64M，推理速度飞快，又兼具大模型博学！


作者提供了 MiniMind MoE Model 的可视化.

![image](./images/LLM-structure-moe.jpg)

### 剖析 MoE 的核心组件

1. 👨‍⚕️ **Experts（专家集合）**：每个专家本质上就是一个结构完全相同的 `FeedForward` 网络，但训练时由于分工不同，各专家的权重会分化，各自擅长不同领域知识。
2. 🧭 **Router / Gate（路由门控）**：一个轻量的线性分类层，看一眼输入的词向量，计算出每个专家的匹配概率，挑选出得分最高的专家（Top-k）。
3. ⚖️ **辅助损失（Auxiliary Loss / 负载均衡损失）**：  
   - **小白痛点**：如果不加约束，网络很容易发生“专家坍塌”——门控发现某一个专家表现好，就疯狂把所有活都派给它，导致其他 3 个专家长期闲置、彻底荒废。  
   - **解法**：在训练 Loss 里额外加一项“劳逸均匀惩罚”，迫使门控网络必须把活均匀地分发给每个专家！


### 积木 7：分诊台——MoE 门控单元（MoE Gate）

#### 🎯 核心目的与作用
负责计算每个 Token 应该路由到哪位专家，并输出路由权重（Softmax）与辅助损失 `aux_loss`。


In [ ]:
class MoEGate(nn.Module):
    def __init__(self, config: LMConfig):
        super().__init__()
        self.config = config
        self.top_k = config.num_experts_per_tok
        self.n_routed_experts = config.n_routed_experts # 总专家数量

        self.scoring_func = config.scoring_func # 评分函数
        self.alpha = config.aux_loss_alpha # 辅助损失的 alpha 参数
        self.seq_aux = config.seq_aux # 是否启用序列辅助损失

        self.norm_topk_prob = config.norm_topk_prob
        self.gating_dim = config.dim
        self.weight = nn.Parameter(torch.empty((self.n_routed_experts, self.gating_dim))) # 混合专家层 -> embeding dim 层
        self.reset_parameter()

    def reset_parameter(self):
        import torch.nn.init as init
        init.kaiming_uniform_(self.weight, a=math.sqrt(5))

    def forward(self, hidden_states):
        bsz, seq_len, h = hidden_states.shape
        hidden_states = hidden_states.view(-1, h) # (total_tokens, hidden_dim) 其中 total_tokens = bsz * seq_len
        logits = F.linear(hidden_states, self.weight, None) # logits = hidden_states · self.weight.T
        # print(f'shape of logits computed: {logits.shape}') # (total_tokens, n_routed_experts)

        if self.scoring_func == 'softmax':
            scores = logits.softmax(dim=-1)
        else:
            raise NotImplementedError(f'insupportable scoring function for MoE gating: {self.scoring_func}')

        topk_weight, topk_idx = torch.topk(scores, k=self.top_k, dim=-1, sorted=False)
        if self.top_k > 1 and self.norm_topk_prob: # prob 归一化
            denominator = topk_weight.sum(dim=-1, keepdim=True) + 1e20
            topk_weight = topk_weight / denominator

        if self.training and self.alpha > 0.0:
            scores_for_aux = scores
            aux_topk = self.top_k
            topk_idx_for_aux_loss = topk_idx.view(bsz, -1) # (bsz, seq_len * num_experts_per_tok)
            if self.seq_aux:
                ################# 对所有批次计算 Experts 重要性得分 #################
                scores_for_seq_aux = scores_for_aux.view(bsz, seq_len, -1)
                ce = torch.zeros(bsz, self.n_routed_experts, device=hidden_states.device)
                # 根据索引将 torch.ones 累加到 ce 中，并执行归一化
                ce.scatter_add_(1, topk_idx_for_aux_loss,
                                torch.ones(bsz, seq_len * aux_topk, device=hidden_states.device)).div_(
                    seq_len * aux_topk / self.n_routed_experts)
                ######################### 计算负载均衡损失 ##########################
                aux_loss = (ce * scores_for_seq_aux.mean(dim=1)).sum(dim=1).mean() * self.alpha
            else:
                mask_ce = F.one_hot(topk_idx_for_aux_loss.view(-1), num_classes=self.n_routed_experts)
                ce = mask_ce.float().mean(0)
                Pi = scores_for_aux.mean(0)
                fi = ce * self.n_routed_experts
                aux_loss = (Pi * fi).sum() * self.alpha
        else:
            aux_loss = 0
        return topk_idx, topk_weight, aux_loss

接下来，我们设置一个 batch size = 4, seq len =16, emb dim = 512 的输入向量，在 MoE 门控单元前向传播进行观察

In [ ]:
# 重新创建一个 LMConfig 使得 use MoE 为 True
LMConfig_MoE = LMConfig(n_layers=2, use_moe=True)

In [ ]:
gate = MoEGate(LMConfig_MoE)
hidden_states = torch.randn((4, 16, 512))
topk_idx, topk_weight, aux_loss = gate(hidden_states)
print(f'shape: topk_idx = {topk_idx.shape}, topk_weight = {topk_weight.shape}, aux_loss = {aux_loss}')
print(f'token 0 选择的专家：idx = {topk_idx[0]}, weight = {topk_weight[0]}')
print(f'辅助损失：aux_loss = {aux_loss}')

In [ ]:
del gate

### 积木 8：专家组——MoE 前馈网络（MoEFeedForward）

#### 🎯 核心目的与作用
在结构上完全取代原本单一的 `FeedForward`。它包含一个 `MoEGate` 门控与 `n_routed_experts` 个专家列表。  
根据门控选出的专家索引，将 Token 分派给对应专家进行计算，最后把专家的输出按门控权重加权求和汇合返回。


In [ ]:
class MoEFeedForward(nn.Module):
    def __init__(self, config: LMConfig):
        super().__init__()
        self.config = config
        # 可选专家层
        self.experts = nn.ModuleList([
            FeedForward(config)
            for _ in range(config.n_routed_experts)
        ])
        self.gate = MoEGate(config)
        # 共享专家层d
        if config.n_shared_experts is not None:
            self.shared_experts = FeedForward(config)

    def forward(self, x):
        identity = x
        orig_shape = x.shape
        bsz, seq_len, _ = x.shape
        # 使用门控机制选择专家
        topk_idx, topk_weight, aux_loss = self.gate(x)
        x = x.view(-1, x.shape[-1])
        flat_topk_idx = topk_idx.view(-1)
        if self.training:
            # 训练模式下，重复输入数据
            x = x.repeat_interleave(self.config.num_experts_per_tok, dim=0)
            y = torch.empty_like(x, dtype=torch.float16)
            # 将输入数据中选择第 i 个专家的部分输入到该专家网络中进行处理，并将输出结果存储到 y 中对应位置的元素
            for i, expert in enumerate(self.experts):
                y[flat_topk_idx == i] = expert(x[flat_topk_idx == i]).to(y.dtype)  #  确保类型一致
            y = (y.view(*topk_weight.shape, -1) * topk_weight.unsqueeze(-1)).sum(dim=1) # 加权求和
            y = y.view(*orig_shape) # 恢复原始形状
        else:
            # 推理模式下，只选择最优专家
            y = self.moe_infer(x, flat_topk_idx, topk_weight.view(-1, 1)).view(*orig_shape)
        if self.config.n_shared_experts is not None:
            y = y + self.shared_experts(identity)
        self.aux_loss = aux_loss
        return y

    @torch.no_grad()
    def moe_infer(self, x, flat_expert_indices, flat_expert_weights):
        expert_cache = torch.zeros_like(x)
        idxs = flat_expert_indices.argsort() # [exp_tok_11, exp_tok_12, exp_tok_21, exp_tok_22] -> idxs[2 * token_idxs] = moe_expert_name
        tokens_per_expert = flat_expert_indices.bincount().cpu().numpy().cumsum(0) #  [1, 3, 1, 2] -> bincount -> [0, 2, 1, 1] -> cumsum -> [0, 2, 3, 4]
        token_idxs = idxs // self.config.num_experts_per_tok
        # 例如当tokens_per_expert=[6, 15, 20, 26, 33, 38, 46, 52]
        # 当token_idxs=[3, 7, 19, 21, 24, 25,  4,  5,  6, 10, 11, 12...]
        # 意味着当token_idxs[:6] -> [3,  7, 19, 21, 24, 25,  4]位置的token都由专家0处理，token_idxs[6:15]位置的token都由专家1处理......
        for i, end_idx in enumerate(tokens_per_expert):
            start_idx = 0 if i == 0 else tokens_per_expert[i - 1]
            if start_idx == end_idx:
                continue
            expert = self.experts[i]
            exp_token_idx = token_idxs[start_idx:end_idx]
            expert_tokens = x[exp_token_idx]
            expert_out = expert(expert_tokens).to(expert_cache.dtype)
            expert_out.mul_(flat_expert_weights[idxs[start_idx:end_idx]]) # 专家网络输出加权
            # 使用 scatter_add_ 进行 sum 操作
            expert_cache.scatter_add_(0, exp_token_idx.view(-1, 1).repeat(1, x.shape[-1]), expert_out)

        return expert_cache

接下来，我们设置一个 batch size = 4, seq len =16, emb dim = 512 的输入向量，在 MoE 门控单元前向传播进行观察

In [ ]:
moe_ffn = MoEFeedForward(LMConfig_MoE).eval()
x = torch.randn((4, 16, 512))
output = moe_ffn(x)
print(f'输出张量：shape = {output.shape}, 辅助损失：aux_loss = {moe_ffn.aux_loss}')

In [ ]:
del moe_ffn

### 积木 9：通用模块——支持 Dense 与 MoE 切换的 MiniMindBlock_DM

通过继承机制，根据配置项 `config.use_moe` 的开关，动态选择装配传统的单个 `FeedForward` 还是多专家的 `MoEFeedForward`：


In [ ]:
# DM stands for Dense & Moe
class MiniMindBlock_DM(MiniMindBlock):
    def __init__(self, layer_id: int, config: LMConfig):
        super().__init__(layer_id, config)
        self.feed_forward = FeedForward(config) if not config.use_moe else MoEFeedForward(config)

In [ ]:
# 检查继承是否正常

miniblock_dm = MiniMindBlock_DM(1, LMConfig_MoE)
print(f'类变量属性检查：layer_id = {miniblock_dm.layer_id}, 类函数属性检查：forward func = {miniblock_dm.forward}')

我们仍然设置一个 batch size = 4, seq len =16, emb dim = 512 的输入向量，在 MoE 门控单元前向传播进行观察.

In [ ]:
miniblock_dm.eval() # 冻结参数，直接 moe infer
x = torch.randn((4, 16, 512))
pos_cis = precompute_pos_cis(64, 16)
output, past_kv = miniblock_dm(x, pos_cis, use_cache=True)
print(f'输出张量：shape = {output.shape}, KV Cache: shape Key = {past_kv[0].shape}, shape Value = {past_kv[1].shape}')
print(f'辅助损失：aux_loss = {miniblock_dm.feed_forward.aux_loss}')

In [ ]:
del miniblock_dm

### 积木 10：整车升级——支持 MoE 的 MiniMindLM_DM 端到端组装

将所有的 Block 全部升级为支持 MoE 路由的模块，组装成能够进行稀疏激活的完整大语言模型！


In [ ]:
class MiniMindLM_DM(MiniMindLM):
    def __init__(self, params: LMConfig = None):
        super().__init__(params)
        structure = 'MoE' if params.use_moe else 'Dense'
        print(f'Initializing MiniMind {structure} Model...\n')
        self.layers = nn.ModuleList([MiniMindBlock_DM(l, params) for l in range(self.n_layers)])

In [ ]:
# 检查一下
a = MiniMindLM_DM(LMConfig_Dense)
b = MiniMindLM_DM(LMConfig_MoE)
del a, b

#### 🔍 动手跟踪：MiniMind MoE 端到端前向推理与辅助损失
传入相同的输入序列 `[1, 3, 5, 7]`，观察 MoE 模型的前向传播，以及随之计算出来的专家负载均衡损失 `aux_loss`：


In [ ]:
MiniMind_MoE = MiniMindLM_DM(LMConfig_MoE)
input_ids = torch.Tensor([1,  3,  5,  7]).long().reshape(1,  4)
OUT = MiniMind_MoE(input_ids,  use_cache=True)
print(f'返回 logits：size = {OUT.logits.shape}, 返回 aux_loss: {OUT.aux_loss},  返回 KV Cache List： len = {len(OUT.past_key_value)}')

In [ ]:
# 我们让 MiniMind 根据我们设计的 四个输入 token 生成输出
out = MiniMind_MoE.generate(input_ids,  max_new_tokens=8,  use_cache=True)
print(f'生成结果：{out}')

In [ ]:
del MiniMind_MoE

## 🎓 学习总结与小白心智模型回顾

恭喜你！到这里，你已经亲手完成了对当前最主流 Decoder-Only 大模型（对齐 LLaMA / Qwen 体系）每一个原子模块的拆解与前向验证！

### 💡 核心知识点卡片回顾
1. **RMSNorm**：水库稳压阀，丢掉均值项轻装上阵，保护深层数值不爆炸；
2. **RoPE**：钟表指针旋转，通过平面角度差天然刻画词与词的相对距离，外推性极佳；
3. **Attention & GQA**：Q 查询、K 标签、V 内容；GQA 让多个 Q 共享 KV，推理显存大降；
4. **KV Cache**：生成下一个词时缓存历史词的 K 和 V，把 $O(N^2)$ 的繁重计算变成 $O(N)$ 的飞速提取；
5. **SwiGLU FFN**：带门控滤波的大脑知识库，负责深层逻辑与事实记忆沉淀；
6. **残差连接**：在每个子层旁边搭直通立交桥，保真原始信息，让百层深度网络的梯度顺畅回传；
7. **MoE**：专业分工按需激活，以小模型的推理计算量享受大模型的博学容量！

---

### 📚 延伸优质参考资料
- [十分钟读懂旋转位置编码（RoPE）](https://www.zhihu.com/tardis/zm/art/647109286?source_id=1003)
- [LLaMA 3.1 架构解读与 GQA 机制解析](https://arxiv.org/abs/2305.13245)
- [混合专家模型 MoE 的全面指南与负载均衡损失推导](https://blog.csdn.net/Code1994/article/details/145209710)
